## Desafio 1 - Leave-One-Instance-Out Bias-Free Protocol

Os resultados estão bastante suspeitos. Voces disseram inicialmente que o leave one instance out tinha resultados superiores aos reportados por Fernandes, mas no artigo estão bem piores (só o LOF que está minimamente aceitável). Eu não estou confiando nesses resultados. Será preciso verificar e provavelmente refazer.

Andei pensando por aqui e acho que tem uma boa estrategia para ser seguida. teremos que refazer os experimentos:

Devemos comparar as metodologias do Ricardo (valores default de hiperparametros) e do Fernandes (valor ajustados nos dados simulados), que separam treino e teste por janelamento da mesma instancia apenas, com nossas duas metodologia: que separa por instancia e que separa  por poco, e ver os impactos. É esperado que a separação por instancia reduza o desempenho dos modelos e que a separação por poço reduza ainda mais.

Os experimentos anteriores devem revelar que há perda de desempenho por conta do bias de similaridade para o problema geral de detecção de falha. O próximo passo, seria computar os resultados para cada falha, para verificar quais falhas são mais suscetíveis ao bias de similaridade.


In [19]:
# =========================================================
# Leave-One-Instance-Out Bias-Free Protocol
# Apenas poços reais (WELL-xxxxx)
# SCRIPT FINAL DEFINITIVO
# =========================================================

from datetime import datetime
start_time = datetime.now()

import pandas as pd
import numpy as np
import warnings
import logging
import re
from pathlib import Path

from tsfresh.feature_extraction import extract_features, MinimalFCParameters
from tsfresh.utilities.dataframe_functions import impute

from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_fscore_support

# ---------------------------------------------------------
# Configurações gerais
# ---------------------------------------------------------
warnings.simplefilter("ignore")
logging.getLogger("tsfresh").setLevel(logging.ERROR)

random_state = 1
np.random.seed(random_state)

data_path = Path("./", "data")

normal_class_code = 0
abnormal_classes_codes = [1, 2, 5, 6, 7, 8]

vars = [
    "P-PDG", "P-TPT", "T-TPT", "P-MON-CKP",
    "T-JUS-CKP", "P-JUS-CKGL", "T-JUS-CKGL", "QGL"
]

columns = ["timestamp"] + vars + ["class"]

sample_size = 3 * 60
min_normal_period_size = 20 * 60

train_frac = 0.6
test_frac = 0.4

max_nan_percent = 0.1
std_vars_min = 0.01

disable_progressbar = True

# ---------------------------------------------------------
# TSFresh
# ---------------------------------------------------------
df_fc_p = MinimalFCParameters()
df_fc_p.pop("sum_values", None)
df_fc_p.pop("length", None)

# ---------------------------------------------------------
# Identificação de poços reais
# ---------------------------------------------------------
def extract_well_number(filename):
    m = re.search(r"WELL-(\d+)", str(filename))
    return int(m.group(1)) if m else None

def is_real_well(filename):
    return extract_well_number(filename) is not None

# ---------------------------------------------------------
# Utilidades
# ---------------------------------------------------------
def class_and_file_generator(data_path):
    for class_path in data_path.iterdir():
        if not class_path.is_dir():
            continue
        class_code = int(class_path.stem)
        for instance_path in class_path.iterdir():
            if instance_path.suffix != ".csv":
                continue
            if not is_real_well(instance_path.name):
                continue
            yield class_code, instance_path

def load_instance(instance_path):
    df = pd.read_csv(instance_path)
    assert list(df.columns) == columns
    return df

def extract_windows(df, indices, label, start_id):
    X_list = []
    y_list = []
    sid = start_id
    for idx in indices:
        if idx + sample_size >= len(df):
            continue
        win = (
            df.iloc[idx:idx + sample_size]
              .drop("class", axis=1)
              .fillna(0)
        )
        win.insert(0, "id", sid)
        X_list.append(win)
        y_list.append({"id": sid, "label": label})
        sid += 1
    if not X_list:
        return pd.DataFrame(), pd.DataFrame(), start_id
    return pd.concat(X_list), pd.DataFrame(y_list), sid

# ---------------------------------------------------------
# Classificadores one-class
# ---------------------------------------------------------
clfs = {
    "OneClassSVM": OneClassSVM(kernel="sigmoid", gamma=0.01, nu=0.001),
    "IsolationForest": IsolationForest(
        n_estimators=200, contamination=0.1, random_state=1
    ),
    "EllipticEnvelope": EllipticEnvelope(
        contamination=0.001, random_state=1
    ),
    "LOF": LocalOutlierFactor(
        n_neighbors=20, novelty=True
    )
}

# ---------------------------------------------------------
# Carrega instâncias reais
# ---------------------------------------------------------
instances = pd.DataFrame(
    class_and_file_generator(data_path),
    columns=["class_code", "instance_path"]
)

instances = instances[instances.class_code.isin(abnormal_classes_codes)]
instances.reset_index(drop=True, inplace=True)

print(f"Instâncias reais utilizadas: {len(instances)}")

# ---------------------------------------------------------
# Leave-One-Instance-Out
# ---------------------------------------------------------
scores = []

for class_code, group in instances.groupby("class_code"):
    print(f"\n### Anomalia {class_code}")
    for test_idx, test_row in group.iterrows():
        print(f"Instância teste: {test_row.instance_path.name}")
      # =========================
        # TESTE – instância i
        # =========================
        df_test = load_instance(test_row.instance_path)
        idx_n = df_test[df_test["class"] == normal_class_code].index
        idx_f = df_test[df_test["class"] == class_code].index
        if len(idx_n) < min_normal_period_size:
            print("  -> ignorada (normal insuficiente)")
            continue
        idx_n = np.random.choice(
            idx_n, int(len(idx_n) * test_frac), replace=False
        )
        global_id = 0
        Xn_test, yn_test, global_id = extract_windows(
            df_test, idx_n, label=1, start_id=global_id
        )
        Xf_test, yf_test, global_id = extract_windows(
            df_test, idx_f, label=-1, start_id=global_id
        )
        if Xn_test.empty and Xf_test.empty:
            print("  -> teste vazio")
            continue
        X_test = pd.concat([Xn_test, Xf_test])
        y_test_df = pd.concat([yn_test, yf_test])
        # =========================
        # TREINO – outras instâncias
        # =========================
        X_train_list = []
        y_train_list = []
        for _, row in group.drop(test_idx).iterrows():
            df_tr = load_instance(row.instance_path)
            idx_n_tr = df_tr[df_tr["class"] == normal_class_code].index
            if len(idx_n_tr) < sample_size:
                continue
            idx_n_tr = np.random.choice(
                idx_n_tr, int(len(idx_n_tr) * train_frac), replace=False
            )
            Xn, yn, global_id = extract_windows(
                df_tr, idx_n_tr, label=1, start_id=global_id
            )
            if not Xn.empty:
                X_train_list.append(Xn)
                y_train_list.append(yn)
        if not X_train_list:
            print("  -> treino vazio")
            continue
        X_train = pd.concat(X_train_list)
        y_train_df = pd.concat(y_train_list)
        # =========================
        # Seleção / normalização
        # =========================
        good_vars = np.isnan(X_train[vars]).mean(0) <= max_nan_percent
        std_vars = np.nanstd(X_train[vars], 0)
        good_vars &= std_vars > std_vars_min
        good_vars = list(good_vars.index[good_vars])
        X_train = X_train[["id", "timestamp"] + good_vars]
        X_test = X_test[["id", "timestamp"] + good_vars]
        scaler = StandardScaler()
        X_train[good_vars] = scaler.fit_transform(X_train[good_vars])
        X_test[good_vars] = scaler.transform(X_test[good_vars])
        # =========================
        # TSFresh
        # =========================
        Xt = extract_features(
            X_train,
            column_id="id",
            column_sort="timestamp",
            default_fc_parameters=df_fc_p,
            impute_function=impute,
            disable_progressbar=disable_progressbar
        )
        Xs = extract_features(
            X_test,
            column_id="id",
            column_sort="timestamp",
            default_fc_parameters=df_fc_p,
            impute_function=impute,
            disable_progressbar=disable_progressbar
        )
        # =========================
        # SANITIZAÇÃO NUMÉRICA
        # =========================
        Xt = Xt.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype("float64")
        Xs = Xs.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype("float64")
        # =========================
        # CLIPPING NUMÉRICO
        # =========================
        MAX_ABS_VALUE = 1e6
        Xt = Xt.clip(-MAX_ABS_VALUE, MAX_ABS_VALUE)
        Xs = Xs.clip(-MAX_ABS_VALUE, MAX_ABS_VALUE)
        y_train = y_train_df.set_index("id").loc[Xt.index]["label"].values
        y_test = y_test_df.set_index("id").loc[Xs.index]["label"].values
        # =========================
        # Treino / Teste
        # =========================
        for name, clf in clfs.items():
            clf.fit(Xt, y_train)
            y_pred = clf.predict(Xs)
            n = min(len(y_test), len(y_pred))
            y_test_a = y_test[:n]
            y_pred_a = y_pred[:n]
            p, r, f1, _ = precision_recall_fscore_support(
                y_test_a, y_pred_a, average="micro"
            )
            scores.append({
                "ANOMALIA": class_code,
                "INSTANCIA": test_row.instance_path.name,
                "POCO": extract_well_number(test_row.instance_path.name),
                "CLASSIFICADOR": name,
                "PRECISAO": p,
                "REVOGACAO": r,
                "F1": f1
            })

# ---------------------------------------------------------
# Resultados
# ---------------------------------------------------------
scores = pd.DataFrame(scores)
scores.to_csv("./results/loio_bias_free_real_wells.csv", index=False)

print("\nMédias por classificador:")
print(scores.groupby("CLASSIFICADOR").mean(numeric_only=True))

print(f"\nTempo total: {datetime.now() - start_time}")

Instâncias reais utilizadas: 52

### Anomalia 1
Instância teste: WELL-00001_20140124213136.csv
  -> ignorada (normal insuficiente)
Instância teste: WELL-00002_20140126200050.csv
  -> ignorada (normal insuficiente)
Instância teste: WELL-00006_20170801063614.csv
Instância teste: WELL-00006_20170802123000.csv
Instância teste: WELL-00006_20180618060245.csv

### Anomalia 2
Instância teste: WELL-00002_20131104014101.csv
Instância teste: WELL-00003_20141122214325.csv
Instância teste: WELL-00003_20170728150240.csv
Instância teste: WELL-00003_20180206182917.csv
  -> ignorada (normal insuficiente)
Instância teste: WELL-00009_20170313160804.csv
Instância teste: WELL-00010_20171218200131.csv
Instância teste: WELL-00011_20140515110134.csv
Instância teste: WELL-00011_20140530100015.csv
  -> ignorada (normal insuficiente)
Instância teste: WELL-00011_20140606230115.csv
Instância teste: WELL-00011_20140720120102.csv
Instância teste: WELL-00011_20140726180015.csv
  -> ignorada (normal insuficiente)
Inst